# Pixel-wise regression

Regression predicts a continuous value for every pixel instead of a class, e.g. water depth, canopy height or biomass. The workflow is the same as for segmentation (see [segmentation.ipynb](segmentation.ipynb)), with a regression dataset and a `RegressionTask`.

The test data has a regression target `target_<i>.tif` for every image `image_<i>.tif`: a float value between 0 and 10 per pixel, with `-9999` where there is no target. Targets don't have to cover the whole image: pixels without a target are left out of the loss and the metrics.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio as rst
from lightning import seed_everything
from lightning.pytorch import Trainer
from matplotlib import pyplot as plt
from rasterio.plot import show

from eotorch.data import (
    RasterDataModule,
    RegressionTask,
    get_regression_dataset,
    splits,
)

DATASET_DIR = Path("tests/data")
NODATA = -9999.0  # value of pixels without a target in the target files

### Create a regression dataset and visualize some samples

`get_regression_dataset` takes the same parameters as `get_segmentation_dataset`, e.g. to match image and label files by date, but there is no class mapping. The targets are read as float32. Pixels without a target are read as NaN and shown in black: these are the pixels with the nodata value declared in the target files, and areas that no target file covers. If your files don't declare their nodata value, pass it as `nodata_value`.

In [ ]:
ds = get_regression_dataset(
    images_dir=DATASET_DIR,
    labels_dir=DATASET_DIR,
    image_glob="image_*.tif",
    label_glob="target_*.tif",
    all_image_bands=("red", "green", "blue"),
    rgb_bands=("red", "green", "blue"),
    cache_size=1,
)
ds.plot_samples(n=1, patch_size=100, show_filepaths=True)

In a Jupyter notebook, the dataset is shown on an interactive map with the footprints of the images, the targets and their intersection.

In [ ]:
ds

### Split the dataset

The splits work the same as for segmentation. Here, one file is held out for validation. See [segmentation.ipynb](segmentation.ipynb) for an AOI-based split.

In [ ]:
train_ds, val_ds = splits.file_wise_split(
    dataset=ds,
    val_img_files=DATASET_DIR / "image_3.tif",
)

### Create a data module for training

The images are normalized on the fly with the per-band mean and standard deviation of the training data. These are saved in the checkpoint, so the same normalization is applied when predicting. The targets are used as they are.

In [ ]:
from eotorch.processing.normalize import band_statistics

seed_everything(43, workers=False)

mean, std = band_statistics(train_ds)
module = RasterDataModule(
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=16,
    patch_size=64,
    mean=mean,
    std=std,
    # num_workers=6,
    # persistent_workers=True,
)

Preview how the training and validation datasets are sampled, to check that they don't overlap. (`count_labels=True` counts the pixels of each class, which is only useful for segmentation.)

In [ ]:
module.preview_data_sampling(max_samples=100)

### Create a task (PyTorch Lightning Module) for regression

The same models as for segmentation can be used: the eotorch models and any architecture of segmentation_models_pytorch (e.g. `model="unet", backbone="resnet34"`).

- `loss`: one of `"mse"`, `"mae"`, `"huber"` and `"smoothl1"`. The Huber loss is less sensitive to outliers than the MSE.
- `num_outputs`: the number of values to predict per pixel, if your targets have several bands.

Pixels without a target (NaN) are left out of the loss and the metrics.

In [ ]:
task = RegressionTask(
    model="deepresunet",
    in_channels=3,
    loss="huber",
    # lr=1e-4,
)

### Create a PyTorch Lightning Trainer

The regression task logs the root mean squared error (`RMSE`), the mean absolute error (`MAE`) and the mean squared error (`MSE`), in the units of the targets. Lower is better, so the checkpoints and early stopping monitor the validation RMSE with `mode="min"`.

In [ ]:
from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)

trainer = Trainer(
    callbacks=[
        LearningRateMonitor(logging_interval="epoch"),
        EarlyStopping(
            monitor="val_RMSE",
            patience=35,
            mode="min",
        ),
        ModelCheckpoint(
            save_last=True,
            monitor="val_RMSE",
            save_top_k=5,
            filename="{epoch}-{val_loss:.4f}-{val_RMSE:.2f}",
            mode="min",
        ),
    ],
    max_epochs=150,
)

### Fit the trainer

In [ ]:
trainer.fit(model=task, datamodule=module)

### Use your model to make predictions

The patch size, the bands and the normalization are read from the checkpoint. The predicted values are written to a float32 GeoTIFF, with NaN as its nodata value. With `blend=True`, overlapping patch predictions are averaged, which avoids visible seams between patches.

In [ ]:
f = trainer.checkpoint_callback.best_model_path
print(f"Using weights {f}")
pred_path = RegressionTask.predict_on_tif_file(
    tif_file_path=DATASET_DIR / "image_3.tif",
    checkpoint_path=f,
    out_file_path="predictions/image_3_pred.tif",
    blend=True,
)
print(pred_path)

### Compare the prediction with the target

`image_3.tif` was held out for validation, so the model hasn't seen its targets. The prediction covers the whole image, also where there is no target. The difference is only shown where there is a target.

In [ ]:
with rst.open(DATASET_DIR / "image_3.tif") as src:
    image = src.read()
with rst.open(DATASET_DIR / "target_3.tif") as src:
    target = np.ma.masked_equal(src.read(1), NODATA)
with rst.open(pred_path) as src:
    prediction = np.ma.masked_invalid(src.read(1))

error = prediction - target
rmse = np.sqrt((error**2).mean())
print(f"RMSE on the pixels with a target: {rmse:.2f}")

# The same color scale for the target and the prediction
vmin, vmax = target.min(), target.max()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
show(image, ax=axes[0], adjust=True, title="Image")
for ax, values, title in [
    (axes[1], target, "Target"),
    (axes[2], prediction, "Prediction"),
]:
    plotted = ax.imshow(values, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    fig.colorbar(plotted, ax=ax, fraction=0.046)
largest_error = np.abs(error).max()
plotted = axes[3].imshow(error, cmap="RdBu_r", vmin=-largest_error, vmax=largest_error)
axes[3].set_title("Prediction - target")
fig.colorbar(plotted, ax=axes[3], fraction=0.046)
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
plt.show()